# Step 3.4: the leak detector

This notebook is the check of step 3.4. A feature leaks when its value at a bar reads data from after that bar's close, as `shift(-1)` does: in a backtest it would know the future. `ff.features.validate` catches it. It computes the feature over a window, then again on the same data cut at many instants: each session's close, just after it, in the break or over the weekend, just before and at the next open, at a roll, inside and at the close of a higher timeframe's bar, and at bars drawn at random. The values before each cut must stay what they were. It also computes the window again from a later start, which a feature that never forgets fails, and reads the feature's code for the usual mistakes, each with its line.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and keeps each report beside the feature's versions in `FEFAT_HOME`.

## A feature that reads the next bar

`neighbour_close` gives each 5-minute bar the close of the bar after it: a leak. The product is MES when your store holds its 1-minute bars, else the product with the most contracts, and the window the two weeks to the last trade date of its data.

Expected: the product and the window; then the report. Its first line ends in `FAILED`. The second names the first cut that changed a value, the close of the window's first session, and the row, that session's last bar, which holds a value with the data after the cut and none without it: the next bar's close, which the report keeps but never prints, since it is a price of your data. The start check agrees, and the lint names line 3, `shift(-1)`, and why it reads ahead.

In [ ]:
from datetime import timedelta

import polars as pl

import fefat as ff

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
FIRST = LAST - timedelta(days=14)
print(f"{ROOT} 5-minute bars from {FIRST} to {LAST}")


@ff.feature(outputs={"neighbour_close": "price"})
def neighbour_close(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("close").shift(-1)


leaky = ff.features.validate(neighbour_close(), ROOT, "5m", FIRST, LAST)
print(leaky.summary())

## The same feature reading the bar before

Fixed to `shift(1)`, the feature reads the bar before each one, which closed before it, and declares the one bar it needs before its first value.

Expected: a report whose first line ends in `PASSED`, no cut failing, the start check agreeing, and no lint finding.

In [ ]:
@ff.feature(warmup=1, outputs={"neighbour_close": "price"})
def neighbour_close(bars: pl.LazyFrame) -> pl.Expr:
    return pl.col("close").shift(1)


fixed = ff.features.validate(neighbour_close(), ROOT, "5m", FIRST, LAST)
print(fixed.summary())